<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_YOURDATA_SuperAnimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

## DeepLabCut 模型库：SuperAnimal 模型

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1616492373700-PGOAC72IOB6AUE47VTJX/ke17ZwdGBToddI8pDm48kB8JrdUaZR-OSkKLqWQPp_YUqsxRUqqbr1mOJYKfIPR7LoDQ9mXPOjoJoqy81S2I8N_N4V1vUb5AoIIIbLZhVYwL8IeDg6_3B-BRuF4nNrNcQkVuAT7tdErd0wQFEGFSnBqyW03PFN2MN6T6ry5cmXqqA9xITfsbVGDrg_goIDasRCalqV8R3606BuxERAtDaQ/modelzoo.png?format=1000w)

# 🦄 DeepLabCut PyTorch 中的 SuperAnimal 模型！ 🔥

本 Notebook 演示了如何在 DeepLabCut 3.0 中使用我们的 SuperAnimal 模型！请阅读 [Ye 等人发表在 Nature Communications 2024 的论文](https://www.nature.com/articles/s41467-024-48792-2) 以了解更多关于 SuperAnimal 模型的信息，并遵循下面的步骤进行操作！

### **开始吧：在 COLAB 中安装最新版本的 DeepLabCut：**

*另外，请确保您已连接到 GPU：前往菜单，点击 Runtime > Change Runtime Type > 选择 "GPU"*

In [ ]:
!pip install --pre deeplabcut

**请务必先点击上方输出中的 "restart runtime"（重启运行时）后再继续操作！**

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import deeplabcut
import deeplabcut.utils.auxiliaryfunctions as auxiliaryfunctions
from deeplabcut.pose_estimation_pytorch.apis import (
    superanimal_analyze_images,
)
from deeplabcut.modelzoo import build_weight_init
from deeplabcut.modelzoo.utils import (
    create_conversion_table,
    read_conversion_table_from_csv,
)
from deeplabcut.modelzoo.video_inference import video_inference_superanimal
from deeplabcut.utils.pseudo_label import keypoint_matching

## 零样本图像和视频推理

SuperAnimal 模型是基础的动物姿态模型。它们可用于**零样本 (zero-shot) 预测**，而无需在特定数据上进行额外训练。

在本节中，我们将展示如何使用 SuperAnimal 模型（给定一个图像文件夹）从图像中预测姿态，并将预测出的图像（带有姿态标注）输出到另一个目标文件夹中。

### 零样本图像推理

如果你有一个要测试的单独图像，请在这里上传它！

#### 上传您想要进行预测的图片

In [ ]:
from google.colab import files

uploaded = files.upload()
for filepath, content in uploaded.items():
    print(f"User uploaded file '{filepath}' with length {len(content)} bytes")
image_path = os.path.abspath(filepath)
image_name = os.path.splitext(image_path)[0]

# If this cell fails (e.g., when using Safari in place of Google Chrome),
# manually upload your video via the Files menu to the left
# and define `image_path` yourself with right click > copy path on the image:
#
# image_path = "/path/to/my/image.png"
# image_name = os.path.splitext(image_path)[0]

#### 选择一个 SuperAnimal 名称和对应的模型架构

请查阅我们的 [SuperAnimals](https://github.com/DeepLabCut/DeepLabCut/blob/main/docs/ModelZoo.md) 文档以了解更多信息！

In [ ]:
# @markdown ---
# @markdown SuperAnimal Configurations
superanimal_name = "superanimal_topviewmouse" #@param ["superanimal_topviewmouse", "superanimal_quadruped"]
model_name = "hrnet_w32" #@param ["hrnet_w32", "resnet_50"]
detector_name = "fasterrcnn_resnet50_fpn_v2" #@param ["fasterrcnn_resnet50_fpn_v2", "fasterrcnn_mobilenet_v3_large_fpn"]

# @markdown ---
# @markdown What is the maximum number of animals you expect to have in an image
max_individuals = 3  # @param {type:"slider", min:1, max:30, step:1}

In [ ]:
# Note you need to enter max_individuals correctly to get the correct number of predictions in the image.
_ = superanimal_analyze_images(
    superanimal_name,
    model_name,
    detector_name,
    image_path,
    max_individuals,
    out_folder="/content/",
    close_figure_after_save=False,
)

### 零样本视频推理

这可以在有或没有视频适配（video adaptation）的情况下完成（速度更快，但未在您的数据上进行自监督微调！）。

#### 上传一个您想要进行预测的视频

In [ ]:
from google.colab import files

uploaded = files.upload()
for filepath, content in uploaded.items():
    print(f"User uploaded file '{filepath}' with length {len(content)} bytes")
video_path = os.path.abspath(filepath)
video_name = os.path.splitext(video_path)[0]

# If this cell fails (e.g., when using Safari in place of Google Chrome),
# manually upload your video via the Files menu to the left
# and define `video_path` yourself with right click > copy path on the video.

#### 选择超级动物（superanimal）和模型名称

In [ ]:
# @markdown ---
# @markdown SuperAnimal Configurations
superanimal_name = "superanimal_topviewmouse" #@param ["superanimal_topviewmouse", "superanimal_quadruped"]
model_name = "hrnet_w32" #@param ["hrnet_w32", "resnet_50"]
detector_name = "fasterrcnn_resnet50_fpn_v2" #@param ["fasterrcnn_resnet50_fpn_v2", "fasterrcnn_mobilenet_v3_large_fpn"]

# @markdown ---
# @markdown What is the maximum number of animals you expect to have in an image
max_individuals = 3  # @param {type:"slider", min:1, max:30, step:1}

#### 无需视频自适应的零样本视频推理

标记后的视频（以及该视频的姿态预测）将保存在 `"/content/"` 目录下，其文件名格式为 `{your_video_name}_superanimal_{superanimal_name}_hrnetw32_labeled.mp4`。

In [ ]:
_ = video_inference_superanimal(
    videos=video_path,
    superanimal_name=superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    video_adapt=False,
    max_individuals=max_individuals,
    dest_folder="/content/",
)

#### 零样本视频推理与视频自适应（无监督）

带标签的视频（以及该视频的姿态预测结果）将保存到 `"/content/"` 目录中，带标签的视频文件名为 `{your_video_name}_superanimal_{superanimal_name}_hrnetw32_labeled_after_adapt.mp4`。

In [ ]:
_ = video_inference_superanimal(
    videos=[video_path],
    superanimal_name=superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    video_adapt=True,
    max_individuals=max_individuals,
    pseudo_threshold=0.1,
    bbox_threshold=0.9,
    detector_epochs=1,
    pose_epochs=1,
    dest_folder="/content/"
)

## 使用 SuperAnimal 进行训练

在本节中，我们将比较在 DeepLabCut 3.0 中训练模型的不同方式，即是否使用经过 SuperAnimal 预训练的模型。您可以对比评估结果，了解每个基线的性能。我们设置了以下基线：

- ImageNet 迁移学习（不使用 SuperAnimal 进行训练）
- SuperAnimal 迁移学习（基线 1）
- SuperAnimal 朴素微调（Baseline 2）
- SuperAnimal 记忆回放微调（Baseline 3）

所有测试均在您现有的 DeepLabCut 项目上进行！如果您目前没有可用于 SuperAnimal 模型的 DeepLabCut 项目，您可以随时使用 DeepLabCut 仓库中[提供的示例开放场数据集](https://github.com/DeepLabCut/DeepLabCut/tree/main/examples/openfield-Pranav-2018-10-30)，或者使用 Zenodo 上[可用的 Tri-Mouse 数据集](https://zenodo.org/records/5851157)。

### 准备 DeepLabCut 项目

首先，将您的 DeepLabCut 项目文件夹放入您的 Google Drive 中！例如，将名为 "Project-YourName-TheDate" 的文件夹移动到 Google Drive 内部。

In [ ]:
# Now, let's link to your GoogleDrive. Run this cell and follow the
# authorization instructions:

from google.colab import drive
drive.mount('/content/drive')

您需要编辑 `config.yaml` 文件中的项目路径，将其设置为您的 Google Drive 链接！

通常，该路径的格式应为：`/content/drive/MyDrive/yourProjectFolderName`。您可以通过以下步骤获取此路径：

1.  转到左侧窗格中的文件导航器。
2.  找到您的 DeepLabCut 项目文件夹。
3.  点击文件夹名称旁边的垂直 `...` 选项。
4.  选择“复制路径 (Copy path)”。

如果挂载驱动器后 `drive` 文件夹不可见，请刷新可用文件列表！

In [ ]:
# TODO: Update the `project_path` to be the path of your DeepLabCut project!
project_path = Path("/content/drive/MyDrive/my-project-2024-07-17")
config_path = str(project_path / "config.yaml")

然后，使用下面的面板选择适合您项目的 `SuperAnimal` 模型（别忘了运行该单元格）！

In [ ]:
# @markdown ---
# @markdown SuperAnimal Configurations
superanimal_name = "superanimal_topviewmouse" #@param ["superanimal_topviewmouse", "superanimal_quadruped"]
model_name = "hrnet_w32" #@param ["hrnet_w32", "resnet_50"]
detector_name = "fasterrcnn_resnet50_fpn_v2" #@param ["fasterrcnn_resnet50_fpn_v2", "fasterrcnn_mobilenet_v3_large_fpn"]

### 不同训练基线之间的比较

数据划分（data split）的定义：**训练图像**和**测试图像**的唯一组合。

我们创建了一个名为 `split 0` 的数据划分。所有基线模型（baselines）将共享此数据划分，以确保比较的公平性。

- `split 0` -> 由所有基线模型共享
- `shuffle 0 (split0)` -> ImageNet 迁移学习
- `shuffle 1 (split0)` -> SuperAnimal 迁移学习
- `shuffle 2 (split0)` -> SuperAnimal 朴素微调（naive fine-tuning）
- `shuffle 3 (split0)` -> SuperAnimal 记忆回放微调（memory-replay fine-tuning）

### 基线（Baselines）之间的区别是什么？

**迁移学习（Transfer learning）**
对于标准的、与任务无关的迁移学习而言，编码器（encoder）会从大规模的预训练数据集中学习通用的视觉特征，然后使用一个随机初始化的解码器（decoder）从下游数据集中学习姿态（pose）。

**微调（Fine-tuning）**
对于面向特定任务的微调而言，编码器和解码器都会在预训练数据集中学习与任务相关的视觉姿态特征，然后在下游数据集中微调解码器，以更新姿态先验（pose priors）。关键在于，网络拥有特定于姿态估计的权重。

**ImageNet 迁移学习（ImageNet transfer-learning）**
编码器仅使用 ImageNet 进行预训练。解码器则在下游任务中从头开始训练。

**SuperAnimal 迁移学习（SuperAnimal transfer-learning）**
编码器首先使用 ImageNet 预训练，然后在我们收集的姿态数据集上进行预训练。之后，解码器在下游任务中从头开始训练。

**SuperAnimal 朴素微调（SuperAnimal naive fine-tuning）**
编码器和解码器都已在预先收集的姿态数据集上进行了预训练。在下游数据集中，我们仅微调那些对应于下游数据集中已标注关键点（keypoints）的卷积通道。这种方法会导致模型对那些在下游数据集中**未被标注**的关键点产生灾难性遗忘（catastrophic forgetting）。

**SuperAnimal 记忆回放微调（SuperAnimal memory-replay fine-tuning）**
如果我们不对 SuperAnimal 进行微调时增加额外处理，模型会忘记那些在下游数据集中未被标注的关键点。为了缓解这个问题，我们混合了 SuperAnimal 模型的标注数据和零样本预测（zero-shot predictions），从而创建一个数据集来“回放” SuperAnimal 关键点的记忆。

In [ ]:
imagenet_transfer_learning_shuffle = 0
superanimal_transfer_learning_shuffle = 1
superanimal_naive_finetune_shuffle = 2
superanimal_memory_replay_shuffle = 3

In [ ]:
deeplabcut.create_training_dataset(
    config_path,
    Shuffles=[imagenet_transfer_learning_shuffle],
    net_type=f"top_down_{model_name}",
    detector_type=detector_name,
    engine=deeplabcut.Engine.PYTORCH,
    userfeedback=False,
)

### ImageNet 迁移学习

历史上，使用 ImageNet 权重的迁移学习策略通常假设预训练模型中不包含任何“动物姿态任务先验知识”（animal pose task priors），这是一种沿袭自先前任务无关迁移学习的范式。

您可以更改您希望训练的 epoch 数量。训练将花费多长时间取决于许多参数，包括数据集中图像的数量、图像的分辨率以及您训练的 epoch 数量。

In [ ]:
# Note we skip the detector training to save time.
# For Top-Down models, the evaluation is by default using ground-truth bounding
#  boxes. But to train a model that can be used to inference videos and images,
#  you have to set detector_epochs > 0.

deeplabcut.train_network(
    config_path,
    detector_epochs=0,
    epochs=50,
    save_epochs=10,
    batch_size=64,  # if you get a CUDA OOM error when training on a GPU, reduce to 32, 16, ...!
    displayiters=10,
    shuffle=imagenet_transfer_learning_shuffle,
)

现在让我们来评估我们训练好的模型的性能。

In [ ]:
deeplabcut.evaluate_network(config_path, Shuffles=[imagenet_transfer_learning_shuffle])

### 使用 SuperAnimal 权重进行迁移学习

首先，我们准备训练数据洗牌（shuffle），以便使用 SuperAnimal 权重进行迁移学习。由于我们已经创建了一个包含所需 train/test 划分的数据洗牌，我们将使用 `deeplabcut.create_training_dataset_from_existing_split` 函数来保留与 ImageNet 迁移学习洗牌中相同的训练/测试索引。

我们指定要使用所选的 SuperAnimal 模型初始化模型权重，但**不保留解码层**（这就是所谓的迁移学习！）。

In [ ]:
weight_init = build_weight_init(
    cfg=auxiliaryfunctions.read_config(config_path), 
    super_animal=superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    with_decoder=False,
)

deeplabcut.create_training_dataset_from_existing_split(
    config_path,
    from_shuffle=imagenet_transfer_learning_shuffle,
    shuffles=[superanimal_transfer_learning_shuffle],
    engine=deeplabcut.Engine.PYTORCH,
    net_type=f"top_down_{model_name}",
    detector_type=detector_name,
    weight_init=weight_init,
    userfeedback=False,
)

然后，我们使用 `SuperAnimal` 权重启动迁移学习（transfer-learning）的训练。

In [ ]:
deeplabcut.train_network(
    config_path,
    detector_epochs=0,
    epochs=50,
    save_epochs=10,
    batch_size=64,  # if you get a CUDA OOM error when training on a GPU, reduce to 32, 16, ...!
    displayiters=10,
    shuffle=superanimal_transfer_learning_shuffle,
)

最后，我们使用从 SuperAnimal 权重迁移学习得到的模型进行评估。

In [ ]:
deeplabcut.evaluate_network(config_path, Shuffles=[superanimal_transfer_learning_shuffle])

### 使用 SuperAnimal 进行微调（不保留全部 SuperAnimal 关键点）

#### 设置权重初始化和数据集

首先，我们进行关键点匹配。此步骤使得理解现有标注与 SuperAnimal 标注之间的对应关系成为可能。此步骤会产生 3 个输出：
- 混淆矩阵 (The confusion matrix)
- 转换表 (The conversion table)
- 整个数据集上的伪预测结果 (Pseudo predictions over the whole dataset)

#### 什么是关键点匹配？

由于 SuperAnimal 模型具有预定义的关键点，而这些关键点可能与您的标注（Annotations）有所不同，因此我们提出了这种算法来最小化模型与数据集之间的差距。我们使用模型对整个数据集执行**零样本推理（zero-shot inference）**。这为每张图像生成了一对预测结果（predictions）和真实标签（ground truth）。

然后，我们将模型预测（2D 坐标）与真实标签之间的匹配问题，转化为一个**二分匹配（bipartite matching）**问题，其中使用**欧几里得距离（Euclidean distance）**作为成对关键点之间的成本（cost）。

接下来，我们使用**匈牙利算法（Hungarian algorithm）**来解决这个匹配问题。因此，对于每张图像，我们最终会得到一个匹配矩阵，其中 `1` 表示匹配成功，`0` 表示不匹配。

由于模型的预测结果在不同图像之间可能存在噪声，我们会将上述匹配矩阵在所有图像上进行平均，然后执行**另一次二分匹配**，最终得到模型与数据集之间的关键点转换表。

请注意，匹配质量的好坏会影响模型的性能，尤其是在零样本设置下。例如，如果标注中的“鼻子（nose）”被错误地映射为关键点“尾巴（tail）”，反之亦然，那么模型将不得不“忘记”（unlearn）与鼻子和尾巴相对应的通道（参见 Mathis 等人的案例研究）。

In [ ]:
keypoint_matching(
    config_path,
    superanimal_name,
    model_name,
    detector_name,
    copy_images=True,
)

conversion_table_path = project_path / "memory_replay" / "conversion_table.csv"
confusion_matrix_path = project_path / "memory_replay" / "confusion_matrix.png"

# You can visualize the pseudo predictions, or do pose embedding clustering etc.
pseudo_prediction_path = project_path / "memory_replay" / "pseudo_predictions.json"

#### 显示混淆矩阵

X 轴列出了现有标注中的关键点（keypoints）。Y 轴列出了 SuperAnimal 关键点空间中的关键点。颜色越深，表示**人工标注**与 SuperAnimal 标注之间的对应关系越强。

In [ ]:
confusion_matrix_image = Image.open(confusion_matrix_path)

plt.imshow(confusion_matrix_image)
plt.axis('off')  # Hide the axes for better view
plt.show()

#### 显示转换表
`gt` 列代表现有数据集中关键点的名称。`MasterName` 代表 SuperAnimal 关键点空间中对应的关键点。

In [ ]:
df = pd.read_csv(conversion_table_path)
df = df.dropna()

df

#### 将转换表添加到项目的 `config.yaml` 文件中

运行关键点匹配后，您可以将转换表添加到项目的 `config.yaml` 文件中，如果认为某些匹配结果有误，也可以对其进行编辑。例如，对于一个包含 4 个标记身体部位（`'snout', 'leftear', 'rightear', 'tailbase'`）的俯视图（top-view）鼠标数据集，映射项目身体部位到 SuperAnimal 身体部位的转换表应如下所示：

```yaml
# 用于微调 SuperAnimal 权重的转换表
SuperAnimalConversionTables:
  superanimal_topviewmouse:
    snout: nose
    leftear: left_ear
    rightear: right_ear
    tailbase: tail_base
```

In [ ]:
create_conversion_table(
    config=config_path,
    super_animal=superanimal_name,
    project_to_super_animal=read_conversion_table_from_csv(
        conversion_table_path
    ),
)

#### 准备用于使用 SuperAnimal 权重进行（朴素的）微调的训练洗牌和权重初始化

然后，当您调用 `build_weight_init` 并设置 `with_decoder=True` 时，您项目中 `config.yaml` 中的转换表将被用于获取对应身体部位（bodyparts）的预测结果。

In [ ]:
weight_init = build_weight_init(
    cfg=auxiliaryfunctions.read_config(config_path), 
    super_animal=superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    with_decoder=True,
)

deeplabcut.create_training_dataset_from_existing_split(
    config_path,
    from_shuffle=imagenet_transfer_learning_shuffle,
    shuffles=[superanimal_naive_finetune_shuffle],
    engine=deeplabcut.Engine.PYTORCH,
    net_type=f"top_down_{model_name}",
    detector_type=detector_name,
    weight_init=weight_init,
    userfeedback=False,
)

#### 使用 SuperAnimal 启动（朴素）微调的训练

In [ ]:
deeplabcut.train_network(
    config_path,
    detector_epochs=0,
    epochs=50,
    save_epochs=10,
    batch_size=64,  # if you get a CUDA OOM error when training on a GPU, reduce to 32, 16, ...!
    displayiters=10,
    shuffle=superanimal_naive_finetune_shuffle,
)

#### 使用（朴素的）SuperAnimal 进行微调后得到的模型评估

In [ ]:
deeplabcut.evaluate_network(
    config_path,
    Shuffles=[superanimal_naive_finetune_shuffle],
)

### 使用 SuperAnimal 进行记忆回放微调（保留完整的 SuperAnimal 关键点）

**灾难性遗忘 (Catastrophic Forgetting)** 描述了持续学习（Continual Learning）中一个经典的问题：模型在学会解决新任务后，会逐渐丧失解决先前任务的能力。

对 `SuperAnimal` 模型进行微调就属于持续学习的范畴：下游数据集可能定义了与模型先前学习到的关键点（keypoints）不同的定义。因此，模型可能会忘记它先前学到的关键点，而只学习目标数据集中定义的那些。

在这种情况下，使用原始数据集和新数据集一起重新训练（retraining）不是一个可行的选项，因为数据集本身可能不容易共享，并且会需要更多的计算资源。

为了应对这一问题，我们将模型的**零样本推理（zero-shot inference）**视为一个存储原始模型知识的“记忆缓冲区”。当我们对 `SuperAnimal` 模型进行微调时，我们将模型预测的关键点替换为**地面实况标注（ground-truth annotations）**，从而实现了新旧知识的混合学习。

零样本预测的质量可能参差不齐，因此我们使用预测的置信度（confidence of prediction）（0.7）作为阈值，以过滤掉低置信度的预测。如果将该阈值设置为 1，则记忆回放微调（memory replay fine-tuning）就退化为“朴素微调”（naive-fine-tuning）。

#### 为使用 SuperAnimal 进行内存回放微调准备训练随机化和权重初始化

In [ ]:
weight_init = build_weight_init(
    cfg=auxiliaryfunctions.read_config(config_path), 
    super_animal=superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    with_decoder=True,
    memory_replay=True,
)

deeplabcut.create_training_dataset_from_existing_split(
    config_path,
    from_shuffle=imagenet_transfer_learning_shuffle,
    shuffles=[superanimal_memory_replay_shuffle],
    engine=deeplabcut.Engine.PYTORCH,
    net_type=f"top_down_{model_name}",
    detector_type=detector_name,
    weight_init=weight_init,
    userfeedback=False,
)

#### 使用 SuperAnimal 启动内存回放微调的训练

In [ ]:
deeplabcut.train_network(
    config_path,
    detector_epochs=0,
    epochs=50,
    save_epochs=10,
    batch_size=64,  # if you get a CUDA OOM error when training on a GPU, reduce to 32, 16, ...!
    displayiters=10,
    shuffle=superanimal_memory_replay_shuffle,
)

#### 使用 SuperAnimal 进行内存回放微调所获得的模型的评估

In [ ]:
deeplabcut.evaluate_network(config_path, Shuffles=[superanimal_memory_replay_shuffle])